## 1. Environnement & Imports

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os, random, time, copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, transforms, models
from sklearn.metrics import (confusion_matrix, precision_score,
                              recall_score, accuracy_score)

print('PyTorch :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device utilisé  :', device)

## 2. Reproductibilité — Seed

In [ ]:
SEED = 42

def set_seed(seed: int) -> None:
    """Fixe tous les seeds pour la reproductibilité."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
print(f'Seed fixé à {SEED}')

## 3. Montage Google Drive & Données

> **Comment obtenir les données ?**
> ```bash
> wget https://s3.amazonaws.com/content.udacity-data.com/nd089/Cat_Dog_data.zip
> unzip Cat_Dog_data.zip
> ```
> Placez ensuite le dossier `Cat_Dog_data/` dans votre Google Drive,
> dans `MyDrive/cat_dog_image_classification/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/cat_dog_image_classification'
os.chdir(BASE_DIR)
DATA_DIR = os.path.join(BASE_DIR, 'Cat_Dog_data')
CKPT_DIR = os.path.join(BASE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Données :', DATA_DIR)
print('Checkpoints :', CKPT_DIR)

## 4. Fonction d'affichage

In [ ]:
def imshow(image, ax=None, title=None, normalize=True):
    """Affiche un tenseur PyTorch (C,H,W) sous forme d'image."""
    if ax is None:
        fig, ax = plt.subplots()
    img = image.numpy().transpose((1, 2, 0))
    if normalize:
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img  = std * img + mean
        img  = np.clip(img, 0, 1)
    ax.imshow(img)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(axis='both', length=0)
    ax.set_xticklabels(''); ax.set_yticklabels('')
    if title: ax.set_title(title)
    return ax

## 5. Chargement des données

### 5.1 Transformations

**Train** : data augmentation (rotation, flip, recadrage aléatoire) pour réduire le sur-apprentissage.  
**Test/Val** : uniquement redimensionnement + centrage, **sans** augmentation (évaluation déterministe).

In [ ]:
# ── Hyperparamètres globaux ──────────────────────────────────────────
BATCH_SIZE  = 32
IMG_SIZE    = 224   # taille standard pour les CNN pré-entraînés
NUM_WORKERS = 2     # à augmenter si votre runtime le permet
VAL_SPLIT   = 0.15  # 15 % des images de train → validation

# ── Normalisation ImageNet (utilisée aussi pour le from-scratch
#    afin que la comparaison soit équitable) ──────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),   # recadrage aléatoire
    transforms.RandomHorizontalFlip(),         # miroir horizontal
    transforms.RandomRotation(15),             # rotation ±15°
    transforms.ColorJitter(brightness=0.2,
                           contrast=0.2),      # variation de luminosité
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print('Transformations définies.')

### 5.2 Split Train / Validation / Test

In [ ]:
from torch.utils.data import random_split, DataLoader

# Chargement complet du train (avec augmentation)
full_train = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'),
                                   transform=train_transforms)
test_data  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),
                                   transform=test_transforms)

# Split train → train / val
n_val   = int(len(full_train) * VAL_SPLIT)
n_train = len(full_train) - n_val
train_data, val_data = random_split(
    full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
# Le val_data hérite des transforms d'augmentation → on les remplace
val_data.dataset = copy.deepcopy(full_train)
val_data.dataset.transform = test_transforms

trainloader = DataLoader(train_data, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
valloader   = DataLoader(val_data,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
testloader  = DataLoader(test_data,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

CLASS_NAMES = full_train.classes  # ['cat', 'dog']
NUM_CLASSES = len(CLASS_NAMES)

print(f'Train : {n_train} images | Val : {n_val} images | Test : {len(test_data)} images')
print(f'Classes : {CLASS_NAMES}')

### 5.3 Aperçu du dataset

In [ ]:
images, labels = next(iter(trainloader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    imshow(images[i], ax=ax, title=CLASS_NAMES[labels[i]], normalize=True)
fig.suptitle('Exemples du jeu d\'entraînement (après augmentation)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Expérience A — CNN *from scratch*

Architecture : **3 blocs convolutifs** (Conv → BN → ReLU → MaxPool → Dropout),
suivis d'un classifieur entièrement connecté.

- **Batch Normalization** : après chaque convolution pour stabiliser l'entraînement et accélérer la convergence.
- **Dropout** : dans le classifieur (p=0.5) et dans les blocs convolutifs (p=0.25) pour la régularisation.

In [ ]:
class CatDogCNN(nn.Module):
    """
    CNN from scratch — 3 blocs convolutifs + classifieur FC.
    
    Bloc i : Conv2d → BatchNorm2d → ReLU → MaxPool2d → Dropout2d
    """
    def __init__(self, num_classes: int = 2, dropout_conv: float = 0.25,
                 dropout_fc: float = 0.5):
        super().__init__()

        # ── Bloc 1 : 3 → 32 canaux ──────────────────────────────────
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),   # BN : stabilise les activations dès le début
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 224 → 112
            nn.Dropout2d(dropout_conv)
        )
        # ── Bloc 2 : 32 → 64 canaux ──────────────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 112 → 56
            nn.Dropout2d(dropout_conv)
        )
        # ── Bloc 3 : 64 → 128 canaux ─────────────────────────────────
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 56 → 28
            nn.Dropout2d(dropout_conv)
        )
        # ── Classifieur FC ───────────────────────────────────────────
        # 128 * 28 * 28 = 100 352 features en entrée
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_fc),    # Dropout fort pour éviter le sur-apprentissage
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)


model_scratch = CatDogCNN(num_classes=NUM_CLASSES).to(device)

# Vérification des dimensions
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(device)
out   = model_scratch(dummy)
print('Sortie du modèle scratch :', out.shape)  # torch.Size([2, 2])

total_params = sum(p.numel() for p in model_scratch.parameters())
print(f'Paramètres totaux : {total_params:,}')

---
## 7. Expérience B — Transfert Learning (ResNet-18)

- **Base** : ResNet-18 pré-entraîné sur ImageNet (features puissantes dès le départ).
- **Stratégie** : on gèle d'abord toutes les couches, puis on fine-tune tout le réseau
  avec un faible LR (approche *progressive fine-tuning*).
- **Tête adaptée** : la `fc` finale est remplacée par `Linear(512 → 2)` + Dropout.

In [ ]:
def build_transfer_model(num_classes: int = 2,
                          dropout_fc: float = 0.4,
                          freeze_backbone: bool = False) -> nn.Module:
    """
    ResNet-18 pré-entraîné avec tête de classification adaptée.
    
    Args:
        freeze_backbone: si True, gèle les couches convolutives
                         (utile en phase 1 d'entraînement).
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Remplacement de la couche de classification finale
    # ResNet-18 : fc = Linear(512, 1000) → on adapte pour 2 classes
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Dropout(dropout_fc),          # régularisation avant la sortie
        nn.Linear(in_features, num_classes)
    )
    return model


model_transfer = build_transfer_model(NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_transfer.parameters())
print(f'Paramètres entraînables : {trainable:,} / {total:,}')

---
## 8. Fonctions d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Entraîne le modèle sur un epoch, retourne loss et accuracy."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds      = torch.max(outputs, 1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader):
    """Évalue sur un loader, retourne loss, accuracy, precision, recall."""
    model.eval()
    criterion     = nn.CrossEntropyLoss()
    running_loss  = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            running_loss  += loss.item() * images.size(0)
            _, preds       = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss  = running_loss / len(loader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    return avg_loss, accuracy, precision, recall


def train_model(model, trainloader, valloader, criterion, optimizer,
                scheduler, num_epochs: int, model_name: str):
    """
    Boucle d'entraînement complète avec :
      - suivi loss / accuracy / precision / recall
      - sauvegarde du meilleur modèle
      - early monitoring
    """
    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  [],
               'val_prec':   [], 'val_rec':  []}

    best_val_acc  = 0.0
    best_weights  = copy.deepcopy(model.state_dict())
    ckpt_path     = os.path.join(CKPT_DIR, f'{model_name}_best.pth')

    print(f'\n{"="*60}')
    print(f'Entraînement : {model_name}')
    print(f'{"="*60}')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer)
        val_loss, val_acc, val_prec, val_rec = evaluate(model, valloader)

        if scheduler is not None:
            scheduler.step()

        # Sauvegarde du meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, ckpt_path)

        # Enregistrement de l'historique
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_prec'].append(val_prec)
        history['val_rec'].append(val_rec)

        elapsed = time.time() - t0
        print(f'Epoch [{epoch:>2}/{num_epochs}] '
              f'| Train loss: {train_loss:.4f}  acc: {train_acc:.3f} '
              f'| Val loss: {val_loss:.4f}  acc: {val_acc:.3f} '
              f'prec: {val_prec:.3f}  rec: {val_rec:.3f} '
              f'| {elapsed:.1f}s')

    print(f'\nMeilleure val acc : {best_val_acc:.4f} → sauvegardé dans {ckpt_path}')
    model.load_state_dict(best_weights)
    return model, history

---
## 9. Expérience A — Entraînement CNN *from scratch*

On teste **SGD** et **Adam** pour comparer leur vitesse de convergence.

In [ ]:
# ── Hyperparamètres Exp. A ───────────────────────────────────────────
EPOCHS_A   = 20
LR_A_SGD   = 0.01
LR_A_ADAM  = 1e-3
criterion  = nn.CrossEntropyLoss()

# ─── Optimiseur 1 : SGD avec momentum ───────────────────────────────
set_seed(SEED)
model_a_sgd  = CatDogCNN(NUM_CLASSES).to(device)
opt_sgd      = optim.SGD(model_a_sgd.parameters(), lr=LR_A_SGD,
                          momentum=0.9, weight_decay=1e-4)
# StepLR : divise le LR par 10 tous les 7 epochs
sched_sgd    = lr_scheduler.StepLR(opt_sgd, step_size=7, gamma=0.1)

model_a_sgd, hist_a_sgd = train_model(
    model_a_sgd, trainloader, valloader,
    criterion, opt_sgd, sched_sgd,
    EPOCHS_A, 'scratch_sgd'
)

In [ ]:
# ─── Optimiseur 2 : Adam ────────────────────────────────────────────
set_seed(SEED)
model_a_adam  = CatDogCNN(NUM_CLASSES).to(device)
opt_adam      = optim.Adam(model_a_adam.parameters(), lr=LR_A_ADAM,
                            weight_decay=1e-4)
sched_adam    = lr_scheduler.StepLR(opt_adam, step_size=7, gamma=0.1)

model_a_adam, hist_a_adam = train_model(
    model_a_adam, trainloader, valloader,
    criterion, opt_adam, sched_adam,
    EPOCHS_A, 'scratch_adam'
)

---
## 10. Expérience B — Entraînement Transfer Learning (ResNet-18)

In [ ]:
EPOCHS_B  = 15
LR_B_SGD  = 5e-3
LR_B_ADAM = 1e-4   # LR plus faible car les poids pré-entraînés sont déjà bons

# ─── Optimiseur 1 : SGD ─────────────────────────────────────────────
set_seed(SEED)
model_b_sgd  = build_transfer_model(NUM_CLASSES).to(device)
opt_b_sgd    = optim.SGD(model_b_sgd.parameters(), lr=LR_B_SGD,
                          momentum=0.9, weight_decay=1e-4)
# CosineAnnealingLR : descente douce du LR sur toute la durée
sched_b_sgd  = lr_scheduler.CosineAnnealingLR(opt_b_sgd, T_max=EPOCHS_B)

model_b_sgd, hist_b_sgd = train_model(
    model_b_sgd, trainloader, valloader,
    criterion, opt_b_sgd, sched_b_sgd,
    EPOCHS_B, 'transfer_sgd'
)

# ─── Optimiseur 2 : Adam ────────────────────────────────────────────
set_seed(SEED)
model_b_adam = build_transfer_model(NUM_CLASSES).to(device)
opt_b_adam   = optim.Adam(model_b_adam.parameters(), lr=LR_B_ADAM,
                           weight_decay=1e-4)
sched_b_adam = lr_scheduler.CosineAnnealingLR(opt_b_adam, T_max=EPOCHS_B)

model_b_adam, hist_b_adam = train_model(
    model_b_adam, trainloader, valloader,
    criterion, opt_b_adam, sched_b_adam,
    EPOCHS_B, 'transfer_adam'
)

---
## 11. Visualisation & Comparaison des courbes

In [ ]:
def plot_history(histories: dict, title: str):
    """
    Trace loss, accuracy, precision et recall pour plusieurs runs.
    histories = {'NomRun': hist_dict, ...}
    """
    metrics = [
        ('train_loss', 'val_loss',  'Loss',      'lower = meilleur'),
        ('train_acc',  'val_acc',   'Accuracy',  'higher = meilleur'),
        (None,         'val_prec',  'Precision', ''),
        (None,         'val_rec',   'Recall',    ''),
    ]
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(title, fontsize=15, fontweight='bold')

    for ax, (train_key, val_key, ylabel, note) in zip(axes, metrics):
        for idx, (name, hist) in enumerate(histories.items()):
            c = colors[idx]
            epochs = range(1, len(hist[val_key]) + 1)
            if train_key and train_key in hist:
                ax.plot(epochs, hist[train_key], '--', color=c,
                        alpha=0.6, label=f'{name} train')
            ax.plot(epochs, hist[val_key], '-', color=c, label=f'{name} val')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel}' + (f' ({note})' if note else ''))
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# ── Courbes Exp. A ──────────────────────────────────────────────────
plot_history(
    {'Scratch SGD': hist_a_sgd, 'Scratch Adam': hist_a_adam},
    'Expérience A — CNN from scratch'
)

# ── Courbes Exp. B ──────────────────────────────────────────────────
plot_history(
    {'Transfer SGD': hist_b_sgd, 'Transfer Adam': hist_b_adam},
    'Expérience B — Transfert Learning (ResNet-18)'
)

# ── Comparaison globale (meilleur de chaque expérience) ─────────────
best_a = max([hist_a_sgd, hist_a_adam], key=lambda h: max(h['val_acc']))
best_b = max([hist_b_sgd, hist_b_adam], key=lambda h: max(h['val_acc']))
plot_history(
    {'Best Scratch': best_a, 'Best Transfer': best_b},
    'Comparaison Scratch vs Transfert Learning'
)

---
## 12. Évaluation finale sur le jeu de test

In [ ]:
results = {}
for name, model in [('Scratch (SGD)',   model_a_sgd),
                     ('Scratch (Adam)',  model_a_adam),
                     ('Transfer (SGD)',  model_b_sgd),
                     ('Transfer (Adam)', model_b_adam)]:
    loss, acc, prec, rec = evaluate(model, testloader)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    results[name] = {'Loss': loss, 'Accuracy': acc,
                     'Precision': prec, 'Recall': rec, 'F1': f1}
    print(f'{name:<20} | Loss={loss:.4f} Acc={acc:.4f} '
          f'Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}')

---
## 13. Matrice de confusion (meilleur modèle)

In [ ]:
def plot_confusion_matrix(model, loader, class_names, title):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            _, preds = torch.max(model(images), 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    # Erreurs typiques
    fp = cm[0, 1]  # chats classés comme chiens
    fn = cm[1, 0]  # chiens classés comme chats
    print(f'Faux positifs (chat → chien) : {fp}')
    print(f'Faux négatifs (chien → chat) : {fn}')


# Meilleur modèle = Transfer Adam (généralement)
best_model = model_b_adam
plot_confusion_matrix(best_model, testloader, CLASS_NAMES,
                      'Matrice de confusion — Meilleur modèle (Transfer Adam)')

### 13.1 Exemples d'erreurs typiques

In [ ]:
best_model.eval()
error_images, error_preds, error_labels = [], [], []

with torch.no_grad():
    for images, labels in testloader:
        outputs       = best_model(images.to(device))
        _, preds      = torch.max(outputs, 1)
        preds_cpu     = preds.cpu()
        wrong         = preds_cpu != labels
        error_images.extend(images[wrong])
        error_preds.extend(preds_cpu[wrong].numpy())
        error_labels.extend(labels[wrong].numpy())
        if len(error_images) >= 8:
            break

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < len(error_images):
        imshow(error_images[i], ax=ax, normalize=True)
        ax.set_title(
            f'Réel: {CLASS_NAMES[error_labels[i]]}\n'
            f'Prédit: {CLASS_NAMES[error_preds[i]]}',
            color='red', fontsize=9
        )
fig.suptitle('Erreurs typiques du meilleur modèle', fontsize=13)
plt.tight_layout()
plt.show()

---
## 14. Sauvegarde & Rechargement du modèle

In [ ]:
# ── Sauvegarde manuelle du meilleur modèle final ─────────────────────
final_path = os.path.join(CKPT_DIR, 'best_model_final.pth')
torch.save({
    'model_state_dict': best_model.state_dict(),
    'arch': 'resnet18_transfer',
    'classes': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'seed': SEED
}, final_path)
print(f'Modèle sauvegardé : {final_path}')

# ── Rechargement depuis le disque ────────────────────────────────────
checkpoint  = torch.load(final_path, map_location=device)
loaded_model = build_transfer_model(NUM_CLASSES).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()
print('Modèle rechargé avec succès.')

# Vérification : mêmes métriques qu'avant la sauvegarde
_, acc, prec, rec = evaluate(loaded_model, testloader)
print(f'Test rechargé — Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}')